<sub>**The Annotated ML Handbook** · Part V: scikit-learn</sub>

# Chapter 20: Preprocessing, feature engineering and pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-5-scikit-learn/20-sklearn-preprocessing.ipynb)

Companion code for the chapter. Every example from the book is here, in order, ready to run. The line-by-line explanations, definitions and discussion are in the book.

Run the setup cell first, then run the cells from top to bottom.

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

## 20.1 Scaling numeric features · *Beginner*

**StandardScaler, MinMaxScaler, RobustScaler**

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

X = np.array([[1.0, 200.0], [2.0, 300.0], [3.0, 250.0], [4.0, 5000.0]])
for scaler in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    Xs = scaler.fit_transform(X)
    print(f"{type(scaler).__name__:<15}", Xs[:, 1].round(2))
sc = StandardScaler().fit(X)
print("learned mean_:", sc.mean_, " scale_:", sc.scale_.round(1))
print(sc.inverse_transform(sc.transform(X))[0])

StandardScaler  [-0.6  -0.55 -0.58  1.73]
MinMaxScaler    [0.   0.02 0.01 1.  ]
RobustScaler    [-0.06  0.02 -0.02  3.82]
learned mean_: [   2.5 1437.5]  scale_: [1.1000e+00 2.0571e+03]
[  1. 200.]


## 20.2 Encoding categorical features · *Beginner*

**One-hot and ordinal encoding**

In [3]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

df = pd.DataFrame({"region": ["north", "south", "east", "south"],
                   "size": ["S", "L", "M", "M"]})
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
print(ohe.fit_transform(df[["region"]]))
print(ohe.get_feature_names_out())
print(ohe.transform(pd.DataFrame({"region": ["west"]})))
ord_enc = OrdinalEncoder(categories=[["S", "M", "L"]])
print(ord_enc.fit_transform(df[["size"]]).ravel())

[[0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]]
['region_east' 'region_north' 'region_south']
[[0. 0. 0.]]
[0. 2. 1. 1.]


## 20.3 Imputing missing values · *Beginner*

**SimpleImputer and KNNImputer**

In [4]:
from sklearn.impute import SimpleImputer, KNNImputer

Xm = np.array([[1.0, 10.0], [np.nan, 12.0], [3.0, np.nan], [4.0, 16.0]])
print(SimpleImputer(strategy="median").fit_transform(Xm))
print(SimpleImputer(strategy="mean", add_indicator=True).fit_transform(Xm))
print(KNNImputer(n_neighbors=2).fit_transform(Xm).round(2))
print(SimpleImputer(strategy="most_frequent").fit_transform(np.array([["a"], ["b"], ["a"], [np.nan]], dtype=object)).ravel())

[[ 1. 10.]
 [ 3. 12.]
 [ 3. 12.]
 [ 4. 16.]]
[[ 1.         10.          0.          0.        ]
 [ 2.66666667 12.          1.          0.        ]
 [ 3.         12.66666667  0.          1.        ]
 [ 4.         16.          0.          0.        ]]
[[ 1.  10. ]
 [ 2.5 12. ]
 [ 3.  13. ]
 [ 4.  16. ]]
['a' 'b' 'a' 'a']


## 20.4 Pipelines: preprocessing + model as one object · *Beginner*

**A simple pipeline**

In [5]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, random_state=0)

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_tr, y_tr)
print("test accuracy:", round(pipe.score(X_te, y_te), 4))
print(pipe.named_steps["scale"].mean_[:3].round(2))
quick = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print([name for name, _ in quick.steps])

test accuracy: 0.958
[14.12 19.27 91.91]
['standardscaler', 'logisticregression']


## 20.5 ColumnTransformer: different processing per column · *Intermediate*

**The standard tabular preprocessing recipe**

In [6]:
from sklearn.compose import ColumnTransformer, make_column_selector

ins = pd.read_csv("insurance.csv")
X = ins.drop(columns="charges")
y = ins["charges"]
num_cols = ["age", "bmi", "children"]
cat_cols = ["sex", "smoker", "region"]

numeric = Pipeline([("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler())])
categorical = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary"))])
pre = ColumnTransformer([
    ("num", numeric, num_cols),
    ("cat", categorical, cat_cols),
], remainder="drop")

Xt = pre.fit_transform(X)
print(Xt.shape)
print(pre.get_feature_names_out())

(1000, 9)
['num__age' 'num__bmi' 'num__children' 'cat__sex_male' 'cat__smoker_yes'
 'cat__region_northeast' 'cat__region_northwest' 'cat__region_southeast'
 'cat__region_southwest']


**Plugging it into a model**

In [7]:
from sklearn.linear_model import Ridge
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)
model = Pipeline([("pre", pre), ("reg", Ridge(alpha=1.0))])
model.fit(X_tr, y_tr)
print("test R2:", round(model.score(X_te, y_te), 4))
new_customer = pd.DataFrame([{"age": 45, "sex": "female", "bmi": np.nan, "children": 2,
                              "smoker": "no", "region": "southwest"}])
print("predicted charges:", model.predict(new_customer).round(0))

test R2: 0.9311
predicted charges: [19086.]


## 20.6 Seeing leakage with your own eyes · *Intermediate*

**Feature selection outside vs inside the pipeline**

In [8]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(100, 5000))
y_noise = rng.integers(0, 2, 100)

X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
wrong = cross_val_score(LogisticRegression(max_iter=1000), X_sel, y_noise, cv=5).mean()

right_pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
right = cross_val_score(right_pipe, X_noise, y_noise, cv=5).mean()
print(f"selection outside CV: {wrong:.2f}   <- looks impressive, entirely fake")
print(f"selection inside CV : {right:.2f}   <- honest: no signal")

selection outside CV: 0.85   <- looks impressive, entirely fake
selection inside CV : 0.48   <- honest: no signal


## 20.7 Feature engineering · *Intermediate*

**Creating features inside a pipeline**

In [9]:
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, KBinsDiscretizer, SplineTransformer

log_tf = FunctionTransformer(np.log1p, feature_names_out="one-to-one")
print(log_tf.fit_transform(np.array([[0.0], [9.0], [99.0]])).ravel().round(3))
poly = PolynomialFeatures(degree=2, include_bias=False)
print(poly.fit_transform([[2, 3]]), poly.get_feature_names_out(["a", "b"]))
bins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf")
print(bins.fit_transform(np.array([[1], [2], [3], [10], [20], [30]])).ravel())
spl = SplineTransformer(n_knots=4, degree=3)
print(spl.fit_transform(np.linspace(0, 1, 5).reshape(-1, 1)).shape)

[0.    2.303 4.605]
[[2. 3. 4. 6. 9.]] ['a' 'b' 'a^2' 'a b' 'b^2']
[0. 0. 1. 1. 2. 2.]
(5, 6)


**A custom transformer class**

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin

class RatioFeatures(BaseEstimator, TransformerMixin):
    """Add the ratio of two columns as a new column."""
    def __init__(self, num="bmi", den="age"):
        self.num, self.den = num, den
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X[f"{self.num}_per_{self.den}"] = X[self.num] / X[self.den]
        return X

print(RatioFeatures().fit_transform(ins[["age", "bmi"]].head(3)).round(3))

   age   bmi  bmi_per_age
0   58  27.9        0.481
1   26  39.4        1.515
2   19  34.6        1.821


## 20.8 Inspecting and displaying pipelines · *Intermediate*

**get_params, set_params and the step__param syntax**

In [11]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
print(list(pipe.get_params().keys())[-6:])
pipe.set_params(logisticregression__C=0.1)
print(pipe.named_steps["logisticregression"].C)
print(pipe)

['logisticregression__penalty', 'logisticregression__random_state', 'logisticregression__solver', 'logisticregression__tol', 'logisticregression__verbose', 'logisticregression__warm_start']
0.1
Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(C=0.1, max_iter=1000))])


## Try it yourself

1. Build a pipeline for the insurance data that log-transforms `charges` as the target with `TransformedTargetRegressor(regressor=..., func=np.log1p, inverse_func=np.expm1)`. Does test R² improve?
2. Add `PolynomialFeatures(2)` to the numeric branch. What happens to the number of features?
3. Repeat the leakage experiment with `k=5` and `k=200`. How does the fake score change?

Worked solutions: [solutions-parts-5-8.ipynb](../../solutions/solutions-parts-5-8.ipynb), section *Chapter 20*.